# Ten-fold cross-validation

Trains the selected configuration on one fold and evaluates `best.pt` on the
test fold. Run folds 0 to 9, one per session, and run the summary cell when all
folds are done. The split is checked against the fingerprint of the split used
in the study.

## Setup

Select a GPU runtime (**Runtime > Change runtime type**). Outputs are written
to Google Drive, so that a run interrupted by the end of the session can be
resumed. The Roboflow API key is read from the Colab secret
`ROBOFLOW_API_KEY` (key icon in the left panel).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/hugocesarleal/grape-segmentation-training /content/grape-segmentation-training
%cd /content/grape-segmentation-training
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/grape-segmentation"
os.makedirs(OUTPUT, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

## Dataset

Version 14 is the export without offline augmentation. The step also converts
the instances exported as boxes into polygons.

In [ ]:
VERSION = 14  # @param {type:"integer"}
!python -m grapeseg.data --version $VERSION --out /content/dataset

## Fold

In [ ]:
FOLD = 0  # @param {type:"integer"}
BATCH = 8  # @param {type:"integer"}
!python cross_validation.py split --dataset /content/dataset --fold $FOLD --out /content/folds --expected c6165896dbcf

## Training and test evaluation

In [ ]:
!python cross_validation.py train --fold $FOLD --folds /content/folds --batch $BATCH --project $OUTPUT/cv

## Resume an interrupted fold

Run the setup, dataset and fold cells again, then this cell. The test fold is
evaluated when the training ends.

In [ ]:
!python cross_validation.py train --fold $FOLD --folds /content/folds --batch $BATCH --project $OUTPUT/cv --resume

## Summary

In [ ]:
!python cross_validation.py summary --results $OUTPUT/cv